# Linear Regression baseline (ADR)

Untuned Linear Regression baseline for Milestone 1. Everything before the model is reused:
- `Data/train.csv` comes from `Preprocessing/preprocess.ipynb` (columns dropped, invalid rows removed) and `Preprocessing/split.ipynb` (time-based split)
- the preprocessing pipeline comes from `Preprocessing/pipeline.py`



In [2]:
import time

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import TimeSeriesSplit, cross_validate
from sklearn.pipeline import Pipeline

from Preprocessing.pipeline import TARGET, check_groups, build_preprocess

RANDOM_STATE = 67
pd.set_option("display.max_columns", None)

train = pd.read_csv("Data/train.csv")
X_train = train.drop(columns=[TARGET])
y_train = train[TARGET]
check_groups(X_train)
print("train:", X_train.shape)

ModuleNotFoundError: No module named 'Preprocessing.pipeline'

Model pipeline

Preprocessing + Linear Regression in one pipeline, so scaling and encoding are refitted on the training part of every fold.

In [ ]:
model = Pipeline([
    ("prep", build_preprocess()),
    ("reg", LinearRegression()),
])

Cross-validation

`train.csv` is sorted by arrival month, so `TimeSeriesSplit` always trains on earlier bookings and validates on the following ones, the same idea as the train/test split.

In [ ]:
cv = TimeSeriesSplit(n_splits=5)
scoring = {"mae": "neg_mean_absolute_error", "mse": "neg_mean_squared_error", "r2": "r2"}

start = time.perf_counter()
cv_res = cross_validate(model, X_train, y_train, cv=cv, scoring=scoring, n_jobs=-1)
cv_time = time.perf_counter() - start

folds = pd.DataFrame({
    "MAE": -cv_res["test_mae"],
    "MSE": -cv_res["test_mse"],
    "RMSE": np.sqrt(-cv_res["test_mse"]),
    "R2": cv_res["test_r2"],
}, index=[f"Fold {i + 1}" for i in range(cv.n_splits)])
display(folds.round(3))

cv_table = folds.agg(["mean", "std"]).T.round(3)
display(cv_table)
print(f"CV time: {cv_time:.1f}s")

Coefficients

The model is fitted on the full training set to inspect what it learned. Numeric features are scaled, so their sizes are comparable. One-hot coefficients are in € per night relative to the other categories.

In [ ]:
model.fit(X_train, y_train)

names = model.named_steps["prep"].get_feature_names_out()
coefs = pd.Series(model.named_steps["reg"].coef_, index=names).sort_values()

top = pd.concat([coefs.head(10), coefs.tail(10)])
fig, ax = plt.subplots(figsize=(8, 6))
ax.barh(top.index, top.values, color=["tab:blue" if v < 0 else "tab:orange" for v in top.values])
ax.axvline(0, color="black", lw=0.8)
ax.set_title("Top 10 negative and positive coefficients")
ax.set_xlabel("Coefficient (€ per night)")
plt.tight_layout()
plt.show()

Interpretation

- MAE is the average miss in € per night, RMSE punishes the big misses more, R² is the share of the price differences the model explains
- Linear Regression assumes every feature adds a fixed amount to the price, it cannot learn interactions (for example a big room in August costing more than the two effects added)
- the next step is tree-based models (Random Forest, XGBoost) and tuning, compared with the same CV